In [ ]:
import numpy as np

def verify_solution(A, x, b):
    """
    Hàm kiểm tra xem x có phải là nghiệm của phương trình Ax=b hay không.
    Xử lí cả 3 trường hợp vô nghiệm, một nghiệm và vô số nghiệm.
    """
    # Chuyển đổi sang numpy array để tính toán
    A_np = np.array(A, dtype=float)
    b_np = np.array(b, dtype=float).reshape(-1, 1) # Đưa về ma trận cột
    n = A_np.shape[1]

    # Ma trận mở rộng [A|b]
    Ab_np = np.hstack([A_np, b_np])
    
    rank_A = np.linalg.matrix_rank(A_np)
    rank_Ab = np.linalg.matrix_rank(Ab_np)
    
    # TRƯỜNG HỢP 1: Vô nghiệm
    if rank_A < rank_Ab:
        return x is None
        
    # TRƯỜNG HỢP 2: Hệ có nghiệm (duy nhất hoặc vô số)
    if rank_A == rank_Ab:
        if x is None: return False
        
        if isinstance(x, list):
            x_np = np.array(x, dtype=float).flatten()
            return np.allclose(A_np @ x_np, b_np.flatten())
            
        if isinstance(x, dict):
            # Kiểm tra nghiệm riêng
            xp = np.array(x.get('particular', []), dtype=float)
            if not np.allclose(A_np @ xp, b_np.flatten()):
                return False
            
            # Kiểm tra các vector cơ sở có thuộc null space của A không
            basis = x.get('basis', [])
            if len(basis) != (n - rank_A): # Kiểm tra số lượng vector cơ sở
                return False
                
            for v in basis:
                v_np = np.array(v, dtype=float)
                if not np.allclose(A_np @ v_np, 0): # Av phải bằng 0
                    return False
            return True
            
    return False

In [ ]:
from gaussian import *

# KỊCH BẢN KIỂM THỬ
test_cases = [
    {
        "name": "Hệ phương trình cơ bản",
        "A": [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]],
        "b": [8, -11, -3]
    },
    {
        "name": "Ma trận đơn vị",
        "A": [[1, 0], [0, 1]],
        "b": [5, 10]
    },
    {
        "name": "Cần hoán đổi dòng",
        "A": [[0, 1, 2], [1, 2, 1], [2, 7, 8]],
        "b": [4, 4, 17]
    },
    {
        "name": "Hệ vô số nghiệm",
        "A": [[1, 2, 3], [4, 5, 6], [7, 8, 9]],
        "b": [6, 15, 24]
    },
    {
        "name": "Hệ vô nghiệm",
        "A": [[1, 1, 1], [1, 1, 1], [2, 1, 3]],
        "b": [1, 2, 5]
    },
    {
        "name": "Ma trận kích thước lớn (4x4)",
        "A": [[1, 2, 0, -1], [2, 3, -1, 0], [0, -1, 2, 1], [-1, 0, 1, 1]],
        "b": [1, 1, 3, 2]
    }
]

for case in test_cases:
    A, b = case["A"], case["b"]
    print(f"--- Đang kiểm tra: {case['name']} ---")
    
    try:
        # Tính nghiệm bằng hàm tự cài
        _, x, _ = gaussian_eliminate(A, b)
        
        # Kiểm chứng với numpy
        is_correct = verify_solution(A, x, b)
        print(f"Kết quả kiểm chứng: {'HỢP LỆ' if is_correct else 'SAI'}")
        
    except Exception as e:
        import traceback
        print(f"Lỗi thực thi tại case '{case['name']}': {e}")
    print("\n")

--- Đang kiểm tra: Hệ phương trình cơ bản ---
Kết quả kiểm chứng: HỢP LỆ


--- Đang kiểm tra: Ma trận đơn vị ---
Kết quả kiểm chứng: HỢP LỆ


--- Đang kiểm tra: Cần hoán đổi dòng (Pivot bằng 0) ---
khong co pivot tai cot 2
hệ không có nghiệm
Kết quả kiểm chứng: HỢP LỆ


--- Đang kiểm tra: Hệ vô số nghiệm (Dependent) ---
khong co pivot tai cot 2
hệ không có nghiệm duy nhất
Lỗi thực thi tại case 'Hệ vô số nghiệm (Dependent)': could not convert string to float: '1*t_3'


--- Đang kiểm tra: Hệ vô nghiệm (Inconsistent) ---
khong co pivot tai cot 2
hệ không có nghiệm
Kết quả kiểm chứng: HỢP LỆ


--- Đang kiểm tra: Ma trận kích thước lớn (4x4) ---
Kết quả kiểm chứng: HỢP LỆ


